In [1]:
import pandas as pd
# Load preprocessed train/val/test splits
X_train_clean = pd.read_csv("csv/X_train_clean.csv")
X_val_clean = pd.read_csv("csv/X_val_clean.csv")
X_test_clean = pd.read_csv("csv/X_test_clean.csv")

y_train = pd.read_csv("csv/y_train.csv").squeeze()
y_val = pd.read_csv("csv/y_val.csv").squeeze()
y_test = pd.read_csv("csv/y_test.csv").squeeze()

def _align_xy(X, y, name):
    if len(X) != len(y):
        common_index = X.index.intersection(y.index)
        X = X.loc[common_index].copy()
        y = y.loc[common_index].copy()
        print(f"Aligned {name}: X={X.shape}, y={y.shape}")
    return X, y

X_train_clean, y_train = _align_xy(X_train_clean, y_train, "train")
X_val_clean, y_val     = _align_xy(X_val_clean, y_val, "val")
X_test_clean, y_test   = _align_xy(X_test_clean, y_test, "test")

print("Loaded:", X_train_clean.shape, X_val_clean.shape, X_test_clean.shape)

Aligned train: X=(29703, 20), y=(29703,)
Aligned val: X=(8247, 20), y=(8247,)
Aligned test: X=(8823, 20), y=(8823,)
Loaded: (29703, 20) (8247, 20) (8823, 20)


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("✓ XGBoost is available")
except Exception as e:
    XGBOOST_AVAILABLE = False
    print(f"⚠️  XGBoost not available: {e}")

# ==============================================================================
# RANDOM FOREST WITH TimeSeriesSplit CV
# ==============================================================================
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

tscv = TimeSeriesSplit(n_splits=5)

cv_scores_r2   = cross_val_score(rf_model, X_train_clean, y_train, cv=tscv, scoring='r2', n_jobs=-1)
cv_scores_mae  = -cross_val_score(rf_model, X_train_clean, y_train, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
cv_scores_rmse = np.sqrt(-cross_val_score(rf_model, X_train_clean, y_train, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1))

print(f"RF CV R²:   {cv_scores_r2.mean():.4f} ± {cv_scores_r2.std():.4f}")
print(f"RF CV MAE:  {cv_scores_mae.mean():.4f} ± {cv_scores_mae.std():.4f}")
print(f"RF CV RMSE: {cv_scores_rmse.mean():.4f} ± {cv_scores_rmse.std():.4f}")

rf_model.fit(X_train_clean, y_train)

y_train_pred_rf = rf_model.predict(X_train_clean)
y_val_pred_rf   = rf_model.predict(X_val_clean)
y_test_pred_rf  = rf_model.predict(X_test_clean)

# ==============================================================================
# XGBOOST WITH TimeSeriesSplit CV
# ==============================================================================
if XGBOOST_AVAILABLE:
    xgb_model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method='hist'
    )

    cv_scores_r2_xgb   = cross_val_score(xgb_model, X_train_clean, y_train, cv=tscv, scoring='r2', n_jobs=-1)
    cv_scores_mae_xgb  = -cross_val_score(xgb_model, X_train_clean, y_train, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_scores_rmse_xgb = np.sqrt(-cross_val_score(xgb_model, X_train_clean, y_train, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1))

    print(f"XGB CV R²:   {cv_scores_r2_xgb.mean():.4f} ± {cv_scores_r2_xgb.std():.4f}")
    print(f"XGB CV MAE:  {cv_scores_mae_xgb.mean():.4f} ± {cv_scores_mae_xgb.std():.4f}")
    print(f"XGB CV RMSE: {cv_scores_rmse_xgb.mean():.4f} ± {cv_scores_rmse_xgb.std():.4f}")

    xgb_model.fit(X_train_clean, y_train)

    y_train_pred_xgb = xgb_model.predict(X_train_clean)
    y_val_pred_xgb   = xgb_model.predict(X_val_clean)
    y_test_pred_xgb  = xgb_model.predict(X_test_clean)

# ==============================================================================
# METRICS HELPER
# ==============================================================================
def calculate_metrics(y_true, y_pred, model_name, set_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100
    print(f"{model_name} - {set_name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

rf_metrics_train = calculate_metrics(y_train, y_train_pred_rf, "RF", "Train")
rf_metrics_val   = calculate_metrics(y_val,   y_val_pred_rf,   "RF", "Val")
rf_metrics_test  = calculate_metrics(y_test,  y_test_pred_rf,  "RF", "Test")

if XGBOOST_AVAILABLE:
    xgb_metrics_train = calculate_metrics(y_train, y_train_pred_xgb, "XGB", "Train")
    xgb_metrics_val   = calculate_metrics(y_val,   y_val_pred_xgb,   "XGB", "Val")
    xgb_metrics_test  = calculate_metrics(y_test,  y_test_pred_xgb,  "XGB", "Test")

✓ XGBoost is available


RF CV R²:   0.6790 ± 0.0742
RF CV MAE:  0.0323 ± 0.0044
RF CV RMSE: 0.0445 ± 0.0055


XGB CV R²:   0.8728 ± 0.0525
XGB CV MAE:  0.0177 ± 0.0049
XGB CV RMSE: 0.0276 ± 0.0059


RF - Train: MAE=0.0170  RMSE=0.0245  R²=0.9168  MAPE=3.36%
RF - Val: MAE=0.0262  RMSE=0.0361  R²=0.7630  MAPE=5.35%
RF - Test: MAE=0.0282  RMSE=0.0378  R²=0.7931  MAPE=5.89%
XGB - Train: MAE=0.0058  RMSE=0.0095  R²=0.9874  MAPE=1.16%
XGB - Val: MAE=0.0108  RMSE=0.0201  R²=0.9265  MAPE=2.21%
XGB - Test: MAE=0.0132  RMSE=0.0222  R²=0.9286  MAPE=2.70%


In [3]:
# Export trained model(s)
import joblib
import pickle

models = {"rf": rf_model}
if XGBOOST_AVAILABLE:
    models["xgb"] = xgb_model

# Pick best by validation RMSE when both are available
if XGBOOST_AVAILABLE:
    best_name, best_model = min(
        models.items(),
        key=lambda item: rf_metrics_val["RMSE"] if item[0] == "rf" else xgb_metrics_val["RMSE"]
    )
else:
    best_name, best_model = "rf", rf_model

joblib.dump(best_model, f"models/{best_name}_model.joblib")
print(f"Saved best model: {best_name}")

# Optional: save all models
for name, model in models.items():
    joblib.dump(model, f"models/{name}_model.joblib")

# Export xgboost model to models folder as a pkl file
if XGBOOST_AVAILABLE:
    with open("models/xgb_model.pkl", "wb") as f:
        pickle.dump(xgb_model, f)
    print("Exported xgboost model to models/xgb_model.pkl")


Saved best model: xgb
Exported xgboost model to models/xgb_model.pkl
